In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv("/content/dataset.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
unique_products = df['product_id'].nunique()

print(unique_products)

In [ ]:
plt.plot(df.groupby('category')['daily_sales'].mean())
plt.xlabel("name of categories")
plt.ylabel("no of quantity")
plt.title("catagory v/s name of categories")

In [ ]:
low_stock = df[df['stock_qty'] < df['reorder_level']]

low_stock

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year



In [ ]:
print("Total rows:", len(df))

print("Unique products:", df['product_id'].nunique())

print("Date range:")
print(df['date'].min(), "to", df['date'].max())

print("\nRecords per product:")
print(df.groupby('product_id').size().describe())

In [ ]:
df['reorder_required'] = (
    df['stock_qty'] <= df['reorder_level']
).astype(int)

In [ ]:
df.info()

In [ ]:

corr_cols = [
    'stock_qty',
    'reorder_level',
    'daily_sales',
    'unit_cost',
    'lead_days',
    'day',
    'day_of_week'
]

# Calculate correlation
correlation = df[corr_cols].corr()

# Plot heatmap
plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    vmin=-1,
    vmax=1
)

plt.title('Correlation Heatmap of Inventory Features')
plt.tight_layout()
plt.show()

In [ ]:


X = df[features].copy()
y = df['daily_sales'].copy()

# Encode categorical columns
X = pd.get_dummies(
    X,
    columns=['product_id', 'category'],
    dtype=int
)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print(X.head())


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Model 1: Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Model 2: Random Forest Regressor
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)
rf_model.fit(X_train, y_train)

# Model 3: XGBoost Regressor
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    objective='reg:squarederror'
)
xgb_model.fit(X_train, y_train)

print("All 3 models trained successfully!")


In [ ]:
import matplotlib.pyplot as plt

# Generate predictions
lr_pred = lr_model.predict(X_test)
rf_pred = rf_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

predictions = {
    'Linear Regression': lr_pred,
    'Random Forest': rf_pred,
    'XGBoost': xgb_pred
}

# Plot each model separately
for name, y_pred in predictions.items():

    plt.figure(figsize=(10, 5))

    plt.plot(
        range(len(y_test)),
        y_test.values,
        label='Actual Sales',
        marker='o'
    )

    plt.plot(
        range(len(y_pred)),
        y_pred,
        label='Predicted Sales',
        marker='x'
    )

    plt.title(f'Actual vs Predicted Sales - {name}')
    plt.xlabel('Test Sample')
    plt.ylabel('Daily Sales')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

models = {
    'Linear Regression': lr_model,
    'Random Forest': rf_model,
    'XGBoost': xgb_model
}

results = []

for name, model in models.items():

    # Training predictions
    train_pred = model.predict(X_train)

    # Testing predictions
    test_pred = model.predict(X_test)

    results.append({
        'Model': name,
        'Train MAE': mean_absolute_error(y_train, train_pred),
        'Test MAE': mean_absolute_error(y_test, test_pred),

        'Train RMSE': np.sqrt(mean_squared_error(y_train, train_pred)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, test_pred)),

        'Train R2': r2_score(y_train, train_pred),
        'Test R2': r2_score(y_test, test_pred)
    })

results_df = pd.DataFrame(results)

# Round scores
results_df.iloc[:, 1:] = results_df.iloc[:, 1:].round(3)

print(results_df.to_string(index=False))
